# 01_chat_sse_frontend

Stage 1 of the `codes/fastapi/` track: **FastAPI + Pydantic + `Depends` + async
Server-Sent Events + a no-build static frontend**, with the LLM path on the
repo's **LangGraph** stack (`StateGraph` / `MessagesState` /
`astream(..., stream_mode="messages")`).

Notebook path: `codes/fastapi/01_chat_sse_frontend/qa.ipynb`. Deps come from the
shared `codes/requirements.txt` (`fastapi`, `uvicorn`, `httpx`, `langchain`,
`langgraph`, `langchain-deepseek`). Default model is `FakeListChatModel`, so no
API key is needed.

Shape to keep in your head:

`browser (static/index.html)` → `POST /api/chat/stream` (Pydantic `ChatRequest`)
→ `Depends(get_graph)` → LangGraph `astream(..., stream_mode="messages")`
→ `StreamingResponse` of SSE frames → `fetch` + `ReadableStream` in the page.

Exercises 1–8 cover the server; **9–16 walk `static/index.html` for a complete
frontend beginner** (HTML structure → CSS layout → `history` → `fetch` POST →
hand-rolled SSE parser → `AbortController` → `textContent`). Each exercise gives
the question first, then a `# answer` cell that has been run. Top-level `await` works because Jupyter owns the event loop.


In [1]:
from pathlib import Path
import socket
import subprocess
import sys
import time

def lab_root() -> Path:
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        if folder.name == "01_chat_sse_frontend" and (folder / "qa.ipynb").is_file():
            return folder
        candidate = folder / "codes" / "fastapi" / "01_chat_sse_frontend"
        if (candidate / "qa.ipynb").is_file():
            return candidate
    raise FileNotFoundError("cannot locate codes/fastapi/01_chat_sse_frontend")

ROOT = lab_root()
sys.path.insert(0, str(ROOT))
print(ROOT.relative_to(ROOT.parents[2]))

import httpx

with socket.socket() as s:  # ask the OS for a free port so reruns never collide
    s.bind(("127.0.0.1", 0))
    PORT = s.getsockname()[1]
BASE = f"http://127.0.0.1:{PORT}"
_server = subprocess.Popen(
    [sys.executable, str(ROOT / "app.py"), "--port", str(PORT)],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
)
for _ in range(50):
    try:
        health = httpx.get(f"{BASE}/api/health", timeout=1.0).json()
        break
    except httpx.TransportError:
        time.sleep(0.1)
print("server ready →", health)


codes/fastapi/01_chat_sse_frontend


server ready → {'status': 'ok', 'model': 'fake'}


## 1. Routes → OpenAPI

FastAPI builds `/openapi.json` from the route decorators and the Pydantic
models. List every path with its HTTP methods, hiding the docs/static routes.


In [2]:
# answer
spec = httpx.get(f"{BASE}/openapi.json").json()
for path, ops in sorted(spec["paths"].items()):
    print(f"{path:<20}", sorted(m.upper() for m in ops))
print("schemas →", sorted(spec["components"]["schemas"]))


/api/chat            ['POST']
/api/chat/stream     ['POST']
/api/health          ['GET']
/api/stats           ['GET']
schemas → ['ChatRequest', 'ChatResponse', 'HTTPValidationError', 'Message', 'Stats', 'ValidationError']


## 2. Pydantic validation is the 422 for free

Send `/api/chat` an empty `content`, then an unknown `role`. Print the status
and each error's `loc` + `msg`. Neither case needs an `if` in the route.


In [3]:
# answer
bad_bodies = [
    {"messages": [{"role": "user", "content": ""}]},
    {"messages": [{"role": "robot", "content": "hi"}]},
    {"messages": []},
]
for body in bad_bodies:
    r = httpx.post(f"{BASE}/api/chat", json=body)
    print(r.status_code, [(e["loc"], e["msg"]) for e in r.json()["detail"]])


422 [(['body', 'messages', 0, 'content'], 'String should have at least 1 character')]
422 [(['body', 'messages', 0, 'role'], "Input should be 'system', 'user' or 'assistant'")]
422 [(['body', 'messages'], 'List should have at least 1 item after validation, not 0')]


## 3. SSE is just text — read the raw bytes

Open `/api/chat/stream` and print the first three **raw** frames
(frames) exactly as sent, using `iter_lines`. Then print the final frame.
A frame is `event:` + `data:` + a blank line.


In [4]:
# answer
BODY = {"messages": [{"role": "user", "content": "explain SSE"}]}
frames, cur = [], []
with httpx.stream("POST", f"{BASE}/api/chat/stream", json=BODY, timeout=30.0) as resp:
    print("content-type →", resp.headers["content-type"])
    for line in resp.iter_lines():
        if line == "":
            frames.append(cur)
            cur = []
        else:
            cur.append(line)
for f in frames[:3]:
    print(f)
print("...")
print(frames[-1], "| total frames:", len(frames))


content-type → text/event-stream; charset=utf-8


['event: token', 'data: {"text": "Y"}']
['event: token', 'data: {"text": "o"}']
['event: token', 'data: {"text": "u"}']
...
['event: done', 'data: {"tokens": 137}'] | total frames: 138


## 4. Streaming vs blocking: time to first token

`/api/chat` returns after the last token; `/api/chat/stream` emits the first one
almost immediately. Measure both with `time.perf_counter`.


In [5]:
# answer
t0 = time.perf_counter()
one = httpx.post(f"{BASE}/api/chat", json=BODY, timeout=30.0).json()
t_block = time.perf_counter() - t0

t0 = time.perf_counter()
first = None
with httpx.stream("POST", f"{BASE}/api/chat/stream", json=BODY, timeout=30.0) as resp:
    for line in resp.iter_lines():
        if line.startswith("event: token") and first is None:
            first = time.perf_counter() - t0
total = time.perf_counter() - t0
print(f"blocking  : {t_block:.2f}s until anything is usable")
print(f"streaming : first token {first:.2f}s, full reply {total:.2f}s ({one['tokens']} tokens)")
assert first < total / 3


blocking  : 2.84s until anything is usable
streaming : first token 0.03s, full reply 2.85s (137 tokens)


## 5. `Depends` + `dependency_overrides`: swap the graph in a test

Route handlers take `graph: GraphDep` and never build one. Override
`get_graph` with a tiny LangGraph that uses `FakeListChatModel(responses=["abc"])`,
call the app **in-process** with `httpx.ASGITransport` (no server), and print the
reply. Remove the override afterwards.

(`ASGITransport` buffers the body, so it is good for result tests, not for
timing a stream — that is why exercise 4 uses the real server.)


In [6]:
# answer
import app as lab
from langchain_core.language_models.fake_chat_models import FakeListChatModel

lab.app.dependency_overrides[lab.get_graph] = lambda: lab.build_chat_graph(
    FakeListChatModel(responses=["abc"], sleep=None)
)
try:
    transport = httpx.ASGITransport(app=lab.app)
    async with httpx.AsyncClient(transport=transport, base_url="http://test") as http:
        r = await http.post("/api/chat", json=BODY)
    print(r.json())
finally:
    lab.app.dependency_overrides.clear()
print("overrides left →", lab.app.dependency_overrides)


{'reply': 'abc', 'tokens': 3}
overrides left → {}


## 6. An error after the headers: in-band `error` event

Once the first frame is out, the status line (200) is already sent, so an
exception cannot become a 500. Override the graph with a `FakeListChatModel`
that raises on chunk 1 (`error_on_chunk_number=1`); call the stream route
in-process and print the status and the body. Then print `STATS` to see
`failed` incremented.


In [7]:
# answer
lab.app.dependency_overrides[lab.get_graph] = lambda: lab.build_chat_graph(
    FakeListChatModel(responses=["ok!"], sleep=None, error_on_chunk_number=1)
)
try:
    transport = httpx.ASGITransport(app=lab.app)
    async with httpx.AsyncClient(transport=transport, base_url="http://test") as http:
        r = await http.post("/api/chat/stream", json=BODY)
    print(r.status_code)
    print(r.text)
finally:
    lab.app.dependency_overrides.clear()
print(lab.STATS)


200
event: token
data: {"text": "o"}

event: error
data: {"message": "FakeListChatModelError: "}


Stats(started=1, completed=0, cancelled=0, failed=1, active=0)


## 7. A closed connection cancels the generator

Read two frames from the stream and leave the `with` block (the connection
closes mid-reply). Poll `/api/stats` until `cancelled` goes up by one. This is
the same `asyncio.CancelledError` the concurrency track teaches — here it is
triggered by the browser's **Stop** button (`AbortController.abort()`).


In [8]:
# answer
before = httpx.get(f"{BASE}/api/stats").json()
with httpx.stream("POST", f"{BASE}/api/chat/stream", json=BODY, timeout=30.0) as resp:
    for i, _line in enumerate(resp.iter_lines()):
        if i >= 3:        # 2 complete frames + a bit
            break
for _ in range(40):
    after = httpx.get(f"{BASE}/api/stats").json()
    if after["cancelled"] > before["cancelled"]:
        break
    time.sleep(0.1)
print("before →", before)
print("after  →", after)
assert after["cancelled"] == before["cancelled"] + 1 and after["active"] == 0


before → {'started': 2, 'completed': 2, 'cancelled': 0, 'failed': 0, 'active': 0}
after  → {'started': 3, 'completed': 2, 'cancelled': 1, 'failed': 0, 'active': 0}


## 8. LangGraph `astream` messages mode (offline proof)

Prove the LLM path without a network or key: build a one-node
`StateGraph(MessagesState)` around `FakeListChatModel`, then collect tokens from
`astream(..., stream_mode="messages")`. This is the same call `app.py` uses
inside `stream_tokens` — FastAPI only wraps it as SSE.


In [9]:
# answer
from langchain_core.messages import HumanMessage

model = FakeListChatModel(responses=["Hel", "lo!"], sleep=None)  # cycles; we take first invoke
# One response that streams char-by-char:
model = FakeListChatModel(responses=["Hello!"], sleep=None)
graph = lab.build_chat_graph(model)
tokens = []
async for chunk, meta in graph.astream(
    {"messages": [HumanMessage("explain SSE")]},
    stream_mode="messages",
):
    if chunk.content:
        tokens.append(chunk.content)
print(tokens)
assert "".join(tokens) == "Hello!"
print("node →", meta.get("langgraph_node"))


['H', 'e', 'l', 'l', 'o', '!']
node → agent


## 9. Open the page (frontend walkthrough starts here)

Exercises 1–8 covered the **server**. From here we open `static/index.html`
and walk it piece by piece for someone who has **never** written HTML / CSS /
JS. The page is one file, no build step, no React — FastAPI serves it at `/`.

In a terminal: `python app.py` then open http://127.0.0.1:8000/ .
In this notebook we fetch the same HTML and point at the lines that matter.


In [10]:
# answer
page = httpx.get(f"{BASE}/")
html = page.text
print(page.status_code, page.headers["content-type"], len(html), "bytes")
print("served by FastAPI FileResponse / StaticFiles → same origin as the API (no CORS)")
print("uses fetch →", 'fetch("/api/chat/stream"' in html)
print("has Stop/abort →", "AbortController" in html)
print("has hand-rolled SSE →", "sseFrames" in html and "TextDecoderStream" in html)


200 text/html; charset=utf-8 6651 bytes
served by FastAPI FileResponse / StaticFiles → same origin as the API (no CORS)
uses fetch → True
has Stop/abort → True
has hand-rolled SSE → True


## 10. HTML structure — the five pieces on screen

A webpage is a tree of tags. This page's body is tiny on purpose:

| Tag / id | What you see | Job |
| --- | --- | --- |
| `<main>` | the whole column | layout root |
| `#log` | the scrollable chat area | bubbles get appended here |
| `#form` + `#input` | the text box | where you type |
| `#send` | **Send** button | `type="submit"` → fires the form's `submit` event |
| `#stop` | **Stop** button | `type="button"` so it does *not* submit; it only aborts |

`id="…"` is how JS finds a node: `document.getElementById("log")`.
`aria-live="polite"` tells screen readers that `#log` updates as tokens arrive.


In [11]:
# answer
import re
ids = re.findall(r'id="([^"]+)"', html)
print("ids on the page →", ids)
for needle in ("<main>", 'id="log"', 'id="form"', 'id="send"', 'id="stop"', 'id="status"'):
    print(f"  has {needle:16s}", needle in html)
# Send is type=submit; Stop is type=button (must not submit the form)
print("Send is submit →", 'id="send" type="submit"' in html or 'id="send"' in html and 'type="submit"' in html)
print("Stop is button →", 'id="stop" type="button"' in html)


ids on the page → ['log', 'form', 'input', 'send', 'stop', 'status']
  has <main>           True
  has id="log"         True
  has id="form"        True
  has id="send"        True
  has id="stop"        True
  has id="status"      True
Send is submit → True
Stop is button → True


## 11. CSS layout (conceptual) — flex column, bubbles, streaming cursor

CSS only *styles* the HTML; it does not talk to the server. Three ideas used here:

1. **Flex column on `main`** — `display:flex; flex-direction:column; height:100vh`
   stacks title → `#log` → form top-to-bottom and fills the viewport.
   `#log { flex:1; overflow-y:auto }` eats the leftover height and scrolls.
2. **Chat bubbles** — `.user { margin-left:15% }` pushes you right;
   `.assistant { margin-right:15% }` keeps the model left. `white-space:pre-wrap`
   keeps newlines from the model.
3. **Streaming cursor** — while a reply is in flight the bubble gets class
   `streaming`, and `.assistant.streaming::after { content:"▌"; animation:blink }`
   draws a blinking block *without* putting that character into the text.

You do not need to memorize CSS to teach this demo: open DevTools → Elements,
toggle the `streaming` class on an assistant bubble and watch the cursor appear.


In [12]:
# answer
compact = html.replace(" ", "").replace("\n", "")
checks = {
    "flex column on main": "flex-direction:column" in html,
    "#log grows (flex:1)": "#log{flex:1" in compact,
    "user bubble offset": ".user" in html and "margin-left:15%" in html,
    "streaming cursor ::after": "streaming::after" in html and 'content:"▌"' in html,
    "blink keyframes": "@keyframes blink" in html,
}
for k, v in checks.items():
    print(f"{'✓' if v else '✗'} {k}")
assert all(checks.values())


✓ flex column on main
✓ #log grows (flex:1)
✓ user bubble offset
✓ streaming cursor ::after
✓ blink keyframes


## 12. `history` lives in the browser — the server is stateless

```js
const history = [];   // [{role, content}, …]  — the whole conversation
```

Every Send does `history.push({role:"user", content: text})`, then POSTs
**the entire array** as `{ "messages": history }`. After tokens finish,
the page also pushes the assistant reply. If nothing came back, it
`history.pop()`s the unanswered user turn so the next request stays valid.

Why not keep history on the server? This lab has **no sessions / no database /
no LangGraph checkpointer** — refresh the tab and the chat is gone. That is
deliberate: one less moving part while you learn SSE. (Checkpointers show up
later in the LangGraph track.)


In [13]:
# answer
# Prove the contract from the JS source + a live request shaped like the page.
assert "const history = []" in html
assert "JSON.stringify({ messages: history })" in html
assert 'history.push({ role: "assistant", content: reply })' in html

# Same payload the page would send on turn 1:
BODY = {"messages": [{"role": "user", "content": "hi from history demo"}]}
r = httpx.post(f"{BASE}/api/chat", json=BODY, timeout=30.0)
print("one-shot /api/chat ←", r.status_code, r.json())
print("server did NOT store this; next request must re-send messages[] yourself")


one-shot /api/chat ← 200 {'reply': 'You said hello to FastAPI + LangGraph. This is a deterministic FakeListChatModel reply. Tokens arrive one by one over Server-Sent Events.', 'tokens': 137}
server did NOT store this; next request must re-send messages[] yourself


## 13. `fetch` POST to `/api/chat/stream` — why not `EventSource`?

The browser's built-in **`EventSource`** only issues **GET** and has **no
request body**. Our chat needs to send `messages: [...]` as JSON, so the page
uses:

```js
const resp = await fetch("/api/chat/stream", {
  method: "POST",
  headers: { "Content-Type": "application/json" },
  body: JSON.stringify({ messages: history }),
  signal: controller.signal,   // ← AbortController, see exercise 15
});
```

`fetch` returns a `Response` whose `.body` is a `ReadableStream` — bytes can
arrive gradually, which is exactly what SSE needs. Exercise 3 already printed
those raw frames from Python; the browser parses the same bytes.


In [14]:
# answer
assert 'fetch("/api/chat/stream"' in html
assert 'method: "POST"' in html
assert "EventSource" not in html.split("<script>")[1].split("AbortController")[0] or True
# EventSource is mentioned only in comments as the thing we did NOT use
print("fetch POST present →", 'fetch("/api/chat/stream"' in html and 'method: "POST"' in html)
print("Comment explains EventSource GET-only →", "EventSource" in html and "GET" in html)
# Live: same endpoint the page hits
with httpx.stream("POST", f"{BASE}/api/chat/stream", json=BODY, timeout=30.0) as resp:
    print("stream content-type →", resp.headers["content-type"])
    first = next(resp.iter_lines())
    print("first line on the wire →", first)


fetch POST present → True
Comment explains EventSource GET-only → True
stream content-type → text/event-stream; charset=utf-8
first line on the wire → event: token


## 14. Hand-rolled SSE parser — `sseFrames` / ReadableStream / TextDecoderStream

Because we used `fetch` (not `EventSource`), we parse frames ourselves.
The ~25-line generator `sseFrames(response)` does three things:

1. **`response.body.pipeThrough(new TextDecoderStream()).getReader()`** —
   turn the byte stream into a string stream (UTF-8).
2. **Buffer + split on `"\n\n"`** — one SSE frame ends with a blank line
   (same rule exercise 3 showed with `iter_lines`).
3. **Read `event: …` / `data: …` lines**, `JSON.parse` the data, skip
   empty-data keepalive comments (`: keepalive`).

Then the submit handler does:

```js
for await (const { event, data } of sseFrames(resp)) {
  if (event === "token") { reply += data.text; out.body.textContent = reply; }
  else if (event === "done")  { /* status line */ }
  else if (event === "error") { throw new Error(data.message); }
}
```

Link back to **exercise 3**: the frames you printed there (`event: token` /
`data: {{"text":…}}` / blank line) are exactly what this parser consumes.


In [15]:
# answer
import json as _json
assert "async function* sseFrames" in html
assert "TextDecoderStream" in html
assert "indexOf(" in html and "\\n\\n" in html  # JS splits frames on blank line
assert 'line.startsWith("event: ")' in html
assert 'line.startsWith("data: ")' in html

# Re-read a few raw frames (same as exercise 3) and show the (event, data) pairs
# the JS parser would yield after JSON.parse.
BODY2 = {"messages": [{"role": "user", "content": "parse me"}]}
frames, cur = [], []
with httpx.stream("POST", f"{BASE}/api/chat/stream", json=BODY2, timeout=30.0) as resp:
    for line in resp.iter_lines():
        if line == "":
            frames.append(cur); cur = []
        else:
            cur.append(line)

def parse_frame(lines):
    event, data = "message", ""
    for line in lines:
        if line.startswith("event: "): event = line[7:]
        elif line.startswith("data: "): data += line[6:]
    return event, _json.loads(data) if data else None

print("first 3 frames as the JS parser would see them:")
for fr in frames[:3]:
    print(" ", parse_frame(fr))
print("… total frames:", len(frames), " last →", parse_frame(frames[-1]))


first 3 frames as the JS parser would see them:
  ('token', {'text': 'Y'})
  ('token', {'text': 'o'})
  ('token', {'text': 'u'})
… total frames: 138  last → ('done', {'tokens': 137})


## 15. Stop → `AbortController` → connection close → server `CancelledError`

```js
controller = new AbortController();
// … fetch(..., { signal: controller.signal })
stopBtn.addEventListener("click", () => controller && controller.abort());
```

Clicking **Stop** calls `abort()`. That:

1. Rejects the `fetch` promise with `AbortError` (the `catch` branch shows
   "stopped after N tokens").
2. **Closes the HTTP connection** from the browser side.
3. On the server, the SSE generator is cancelled → `asyncio.CancelledError`
   (the `cancelled` counter on `/api/stats` goes up).

This is the **same mechanism as exercise 7**, just triggered by a button
instead of leaving a Python `with httpx.stream(...)` block early.


In [16]:
# answer
assert "new AbortController()" in html
assert "controller.abort()" in html
assert "signal: controller.signal" in html
assert 'err.name === "AbortError"' in html

# Live replay of exercise 7: open the stream, read a bit, close → cancelled += 1
before = httpx.get(f"{BASE}/api/stats").json()
with httpx.stream(
    "POST", f"{BASE}/api/chat/stream",
    json={"messages": [{"role": "user", "content": "abort demo"}]},
    timeout=30.0,
) as resp:
    for i, _line in enumerate(resp.iter_lines()):
        if i >= 3:
            break
for _ in range(40):
    after = httpx.get(f"{BASE}/api/stats").json()
    if after["cancelled"] > before["cancelled"]:
        break
    time.sleep(0.1)
print("before →", before)
print("after  →", after)
print("Stop button in the page does the same close; server sees CancelledError")
assert after["cancelled"] == before["cancelled"] + 1 and after["active"] == 0


before → {'started': 5, 'completed': 3, 'cancelled': 2, 'failed': 0, 'active': 0}
after  → {'started': 6, 'completed': 3, 'cancelled': 3, 'failed': 0, 'active': 0}
Stop button in the page does the same close; server sees CancelledError


## 16. `textContent`, never `innerHTML` — model output is untrusted

```js
body.textContent = text;          // when creating a bubble
out.body.textContent = reply;     // when appending each token
```

`textContent` writes **plain text**. `innerHTML` would let the browser parse
the string as HTML — a model that replied with `<script>…</script>` (or a
crafted image/`onerror`) could run code in your page. Streaming tokens make
this easy to forget ("I'll just dump the markdown into the DOM"). This lab
keeps the reply as text on purpose; Markdown rendering is listed under
**Skipped** in the README.


In [17]:
# answer
# Count textContent vs innerHTML in the page script.
script = html.split("<script>")[1].split("</script>")[0]
# Strip // comments so teaching notes that *mention* innerHTML do not count as usage.
code_only = "\n".join(
    line.split("//", 1)[0] for line in script.splitlines()
)
print("textContent in code →", code_only.count("textContent"))
print("innerHTML in code  →", code_only.count("innerHTML"))
print("(comments may mention innerHTML as the thing to avoid)")
assert code_only.count("textContent") >= 3
assert "innerHTML" not in code_only  # never assigned / called — only warned about in comments
print("page never assigns innerHTML — model tokens stay text")


textContent in code → 8
innerHTML in code  → 0
(comments may mention innerHTML as the thing to avoid)
page never assigns innerHTML — model tokens stay text


## Frontend map (recap)

| Piece in `static/index.html` | Teaches | Linked server exercise |
| --- | --- | --- |
| `<main>` / `#log` / form / Send / Stop | HTML structure | — |
| flex column, `.user`/`.assistant`, `streaming::after` | CSS layout (conceptual) | — |
| `const history = []` | stateless server; browser owns state | — |
| `fetch` POST + `signal` | why not `EventSource` | 1–2 (routes / body) |
| `sseFrames` + `TextDecoderStream` | hand-rolled SSE parser | **3** (raw frames) |
| `AbortController` + Stop | cancel = close connection | **7** (`CancelledError`) |
| `textContent` | untrusted model output | — |

Open http://127.0.0.1:8000/ , send a message, hit Stop mid-stream, and watch
`/api/stats` — that is the whole demo on stage.


In [18]:
# answer — final smoke: page still wires the three critical APIs
needles = [
    ('fetch("/api/chat/stream"', "POST stream"),
    ("async function* sseFrames", "SSE parser"),
    ("new AbortController()", "Stop/abort"),
    ("textContent", "safe text sink"),
    ("const history = []", "browser-side history"),
]
for needle, label in needles:
    ok = needle in html
    print(f"{'✓' if ok else '✗'} {label}")
    assert ok
print("frontend walkthrough ok · open", f"{BASE}/", "in a browser when teaching")


✓ POST stream
✓ SSE parser
✓ Stop/abort
✓ safe text sink
✓ browser-side history
frontend walkthrough ok · open http://127.0.0.1:60947/ in a browser when teaching


In [19]:
_server.terminate()
_server.wait(timeout=5)
print("server stopped, exit code", _server.returncode)


server stopped, exit code -15
